# 02 · Diccionario de datos
Volumen, tipos, formatos, granularidad y periodo histórico, contrastado con el diagrama del levantamiento inicial.

La lógica vive en `scripts/diccionario_datos.py`; aquí solo se ejecuta y se muestra.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../scripts").resolve()))
import pandas as pd
import diccionario_datos as dd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cols, tab, integ, tablas = dd.construir()
hall = dd.hallazgos(tablas)

## 1. Volumen, granularidad y periodo histórico

In [2]:
tab.drop(columns=["tamano_kb"])

,tabla,filas,columnas,granularidad,clave_primaria,columna_periodo,desde,hasta,dias_cubiertos,filas_duplicadas_exactas,filas_con_pk_repetida
0,clientes,53,6,1 fila por cliente,cliente_id,fecha_vinculacion,2015-12-08,2023-04-20,2691,3,6
1,creditos,65,9,1 fila por crédito,credito_id,fecha_desembolso,2023-05-29,2024-05-27,365,0,0
2,pagos,1235,7,1 fila por pago registrado,pago_id,fecha_pago,2023-06-26,2025-12-30,919,5,10
3,gestiones_cobranza,945,7,1 fila por gestión de cobranza,gestion_id,fecha_gestion,2024-07-01,2025-12-31,549,0,0
4,cartera_diaria,35695,8,1 fila por crédito y día (snapshot diario),credito_id + fecha_corte,fecha_corte,2024-07-01,2025-12-31,549,10,20


## 2. Integridad: PK, FK y NOT NULL
`filas_que_incumplen > 0` indica que el dato real no cumple lo que dice el diagrama.

In [3]:
integ

,tabla,regla,filas_que_incumplen,valores
0,clientes,PK única (cliente_id),6,
1,creditos,PK única (credito_id),0,
2,creditos,FK cliente_id → clientes.cliente_id,2,C99999
3,creditos,NOT NULL cliente_id,0,
4,pagos,PK única (pago_id),10,
5,pagos,FK credito_id → creditos.credito_id,0,
6,pagos,NOT NULL credito_id,0,
7,gestiones_cobranza,PK única (gestion_id),0,
8,gestiones_cobranza,FK credito_id → creditos.credito_id,0,
9,gestiones_cobranza,NOT NULL credito_id,0,


## 3. Tipos: diagrama vs datos reales

In [4]:
cols[cols["coincide_tipo"] == "NO"][["tabla", "columna", "tipo_diagrama", "tipo_real", "formato"]]

,tabla,columna,tipo_diagrama,tipo_real,formato
17,pagos,fecha_pago,date,varchar,patrón 9999-99-99 (+1 variantes)
19,pagos,fecha_registro,timestamp,date,YYYY-MM-DD
24,gestiones_cobranza,fecha_gestion,timestamp,date,YYYY-MM-DD


## 4. Hallazgos de calidad

In [5]:
hall[hall["filas_afectadas"] > 0]

,tabla,hallazgo,filas_afectadas,impacto
0,clientes,Filas duplicadas exactas (cliente_id repetido),3,Inflan el conteo de clientes
1,clientes,rango_ingreso nulo,4,Falta una variable de segmentación
2,clientes,antiguedad_laboral_meses negativa,2,Valor imposible
3,clientes,Clientes sin ningún crédito,1,Sin historia de crédito
4,creditos,cliente_id inexistente en clientes (huérfano),2,Rompe la relación cliente-crédito
5,creditos,monto_original <= 0,1,Valor imposible
6,creditos,estado_actual con un único valor (ACTIVO),65,No hay créditos cancelados/castigados: sin variación para el modelo
7,pagos,Filas duplicadas exactas (pago_id repetido),5,Doble conteo de recaudo
8,pagos,fecha_pago en formato dd/mm/yyyy (resto es yyyy-mm-dd),2,Formato mixto: se pierde al convertir a fecha
9,pagos,monto_pagado nulo,3,Pago sin valor


## 5. Diccionario por columna

In [6]:
cols[["tabla", "columna", "clave", "tipo_diagrama", "tipo_real", "formato", "pct_nulos", "distintos", "rango"]]

,tabla,columna,clave,tipo_diagrama,tipo_real,formato,pct_nulos,distintos,rango
0,clientes,cliente_id,PK,varchar,varchar,patrón A99999,0.00,50,C00001 … C00050
1,clientes,fecha_vinculacion,,date,date,YYYY-MM-DD,0.00,49,2015-12-08 … 2023-04-20
2,clientes,tipo_cliente,,varchar,varchar,"categórica: DEPENDIENTE, INDEPENDIENTE",0.00,2,DEPENDIENTE … INDEPENDIENTE
3,clientes,rango_ingreso,,varchar,varchar,"categórica: 1-2M, 2-3M, 3-5M, 5-8M, 8M+",7.55,5,1-2M … 8M+
4,clientes,departamento,,varchar,varchar,"categórica: ANTIOQUIA, ATLANTICO, CUNDINAMARCA, SANTANDER, VALLE",0.00,5,ANTIOQUIA … VALLE
5,clientes,antiguedad_laboral_meses,,int,int,numérico sin separador de miles,0.00,42,-163 … 175
6,creditos,credito_id,PK,varchar,varchar,patrón AA999999,0.00,65,CR000001 … CR000065
7,creditos,cliente_id,FK→clientes.cliente_id,varchar,varchar,patrón A99999,0.00,50,C00001 … C99999
8,creditos,producto,,varchar,varchar,"categórica: CONSUMO, LIBRE_INVERSION",0.00,2,CONSUMO … LIBRE_INVERSION
9,creditos,fecha_desembolso,,date,date,YYYY-MM-DD,0.00,57,2023-05-29 … 2024-05-27


## 6. Exportar a `docs/`

In [7]:
cols.to_csv("../../docs/diccionario_columnas.csv", index=False, encoding="utf-8-sig")
tab.to_csv("../../docs/resumen_tablas.csv", index=False, encoding="utf-8-sig")
integ.to_csv("../../docs/integridad.csv", index=False, encoding="utf-8-sig")
hall.to_csv("../../docs/hallazgos.csv", index=False, encoding="utf-8-sig")
(Path("../../docs") / "diccionario_datos.md").write_text(dd.a_markdown(cols, tab, integ, hall), encoding="utf-8")
print("Exportado en docs/")

Exportado en docs/
